In [ ]:
cd /home/dungh129/recycle_project/deploy

In [1]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from picamera2 import Picamera2
from PIL import Image
import torch.nn as nn
import cv2
import numpy as np


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- Load Full Model --------
#model = models.mobilenet_v2(weights=False)
model = torch.load("mobilenetv2_full.pth", map_location=device, weights_only=False)
#model.load_state_dict(torch.load("mobilenetv2_full.pth", map_location=device))
# model.load_state_dict(torch.load("mobilenetv2_full.pth", map_location=device, weights_only=False))
# model.eval()
print("Model loaded successfully.")

Model loaded successfully.


In [2]:


# -------- Define Transform --------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])



In [3]:
# Import necessary libraries 
import os                        # To handle directory and file operations
import cv2                       # OpenCV library for image processing and GUI display
import time                      # To add delays and timing
import subprocess                # To run terminal commands (for rclone uploads)
from datetime import datetime    # To timestamp filenames to avoid duplicates
from picamera2 import Picamera2  # To interface with Raspberry Pi Camera Module 3
from PIL import Image
from picamera2 import Picamera2

cam = Picamera2()   # Create camera object
cam.start()         # Start the camera stream
time.sleep(2)   
confidence_threshold = 0.8  # 80%

print("Starting bottle classifier... Press 'ESC' to quit.")
# Define bounding box (hard-coded)
bbox = (100, 120, 400, 450)  # (x1, y1, x2, y2)
class_labels = {0: 'Aluminum', 1: 'Background', 2: 'Glass', 3: 'Plastic'}

while True:
    frame = cam.capture_array()
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)  # 🔧 FIX COLOR SHIFT
    frame = cv2.rotate(frame, cv2.ROTATE_180)

    # Crop the ROI
    x1, y1, x2, y2 = bbox
    roi = frame[y1:y2, x1:x2]

    # Center crop
    h, w = frame.shape[:2]
    min_dim = min(h, w)
    x, y = (w - min_dim) // 2, (h - min_dim) // 2
    cropped = frame[y:y+min_dim, x:x+min_dim]

    # Convert to PIL Image
    cropped_rgb = cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB)
    cropped_pil = Image.fromarray(cropped_rgb)
    input_tensor = transform(cropped_pil).unsqueeze(0)

    # Predict
    # label = "Not detecting"
    gray_frame = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)
    mean_pixel_value = gray_frame.mean()

    if mean_pixel_value > 240:  # Adjust threshold if needed
        label = "Not detecting (blank frame)"
    else:
        with torch.no_grad():
            outputs = model(input_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)[0]
            conf, pred = torch.max(probs, 0)
            conf_value = conf.item()
            pred_class = class_labels[pred.item()]

            # Apply your logic:
            if conf_value >= confidence_threshold:
                label = f"{pred_class}"
            elif conf_value > 0:
                label = f"Analyzing: {pred_class} ({conf_value*100:.1f}%)"
            else:
                label = "Not detecting"

    # Display label on the frame
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(frame, label, (x1,y1-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36, 255, 12), 2)

    cv2.imshow("Bottle Classifier", frame)

    # Press 'ESC' to quit
    if cv2.waitKey(1) == 27:
        break

cam.stop()
#cam.release()
cv2.destroyAllWindows()
print(" Camera closed.")


[7:05:40.985168392] [4658]  INFO Camera camera_manager.cpp:326 libcamera v0.5.0+59-d83ff0a4
[7:05:40.997037024] [4676]  INFO RPI pisp.cpp:720 libpisp version v1.2.1 981977ff21f3 29-04-2025 (14:13:50)
[7:05:41.015113233] [4676]  INFO RPI pisp.cpp:1179 Registered camera /base/axi/pcie@1000120000/rp1/i2c@88000/imx708@1a to CFE device /dev/media3 and ISP device /dev/media1 using PiSP variant BCM2712_C0
[7:05:41.025268934] [4658]  INFO Camera camera.cpp:1205 configuring streams: (0) 640x480-XBGR8888 (1) 1536x864-BGGR_PISP_COMP1
[7:05:41.026419605] [4676]  INFO RPI pisp.cpp:1483 Sensor: /base/axi/pcie@1000120000/rp1/i2c@88000/imx708@1a - Selected sensor format: 1536x864-SBGGR10_1X10 - Selected CFE format: 1536x864-PC1B


Starting bottle classifier... Press 'ESC' to quit.
 Camera closed.
